# 第 04 章：高级工具调用 (Smart Tooling)

根据讲义，本章你将掌握以下核心能力：
- **Pydantic v2 契约建模**：建立坚不可摧的参数校验层。
- **运行时上下文注入**：利用 `InjectedState` 实现安全权限透传。
- **防御式编程实操**：应对小模型的参数生成幻觉。

## 1. 统一模型声明
配置环境变量，并加载具备工具调用能力的 DeepSeek 推理基座。

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

# 我们直接沿用项目标准的初始化方式
llm = ChatOpenAI(
    model="deepseek-chat", 
    api_key=os.getenv("DEEPSEEK_API_KEY"), 
    base_url="https://api.deepseek.com"
)
print("模型实例连接就绪。")

## 2. 契约化建模对比：docstring vs Pydantic
通过 Pydantic 显式定义 `args_schema`，让 LLM 面对复杂约束（如数值范围）时表现更稳定。

In [ ]:
from langchain_core.tools import tool
from pydantic import BaseModel, Field
import json

# 推荐范式：显式契约声明
class SearchArgs(BaseModel):
    query: str = Field(description="搜索关键词")
    limit: int = Field(default=5, description="结果数量", ge=1, le=10)

@tool(args_schema=SearchArgs)
def smart_search(query: str, limit: int = 5):
    """
    增强型搜索工具，具有极致的参数校验稳定性。
    """
    return f"[执行搜索] 关键词: {query}, 数量上限: {limit}"

print("--- LLM 接收到的 JSON Schema --- ")
print(json.dumps(smart_search.args_schema.model_json_schema(), indent=2, ensure_ascii=False))

## 3. 安全穿透：使用 InjectedState 注入上下文
演示如何让 Agent 只能看到 `item_id`，而系统自动静默补全敏感的 `user_id`。

In [ ]:
from typing import Annotated
from langgraph.prebuilt import InjectedState

@tool
def secure_delete(item_id: str, user_id: Annotated[str, InjectedState("user_id")]):
    """
    安全删除指定项目。LLM 严禁生成 user_id，该字段由系统自动注入。
    """
    return f"[审计成功] 用户 {user_id} 已发起对项目 {item_id} 的删除操作。"

print("LLM 视界：")
print(f"可见参数: {list(secure_delete.get_input_schema().model_json_schema()['properties'].keys())}")
print(f"隐藏注入: user_id (InjectedState)")

## 4. 场景演练：防御式参数处理
模拟一个幻觉场景：由于模型较小，它生成了一个不存在的字段 `debug_mode`。我们将展示如何提取合法参数。

In [ ]:
from langchain_core.messages import ToolCall

def defensive_executor(tool_call: ToolCall):
    """
    教科书级拦截策略：根据 Schema 静态字段进行参数清洗。
    """
    expected_fields = smart_search.args_schema.model_fields.keys()
    safe_args = {k: v for k, v in tool_call["args"].items() if k in expected_fields}
    
    print(f"原始 Tool Call 参数: {tool_call['args']}")
    print(f"清洗后安全参数: {safe_args}")
    return smart_search.invoke(safe_args)

mock_call = {"name": "smart_search", "args": {"query": "DeepSeek", "limit": 3, "extra_hallucination": "Error"}, "id": "123"}
defensive_executor(mock_call)